# ML-08 — Capstone Modeling Lane (Week 5)

Lane: **Refresh / Content Opportunity Scoring**. Build the right model for the lane,
compare it against the Week-4 baseline on the SAME split and metric, and read the errors.
Simple words, honest numbers.

## 1. Method choice and why

The lane question is a **ranking** question: *"which pages should we refresh first?"*
Ranking needs scores, not labels — so: any classifier's probability, evaluated at precision@K.

- **Logistic regression** first: the readable start. If the signal is linear, this is enough.
- **Decision tree (depth 5)**: still readable — you can print it and explain any single ranking.
- **Random forest**: the stronger ensemble. Added only because the comparison earns it.

Simplicity is a feature: the tree stays as the explainable fallback even though the forest wins.

## 2. Split design

**Client-holdout**, seed 42: 20% of clients (6 of 32) held out entirely — 27,675 train / 2,325 test rows.
Why grouped: in real use the queue scores *new* clients, so the test set must be clients the model never saw.
A naive random row split would leak client patterns across train/test and flatter the numbers (shown in ML-09).

In [ ]:
print('split: client_holdout | seed=42 | 6 of 32 clients held out')
print('train: 27,675 rows | test: 2,325 rows | test base rate (declining): 0.391')

split: client_holdout | seed=42 | 6 of 32 clients held out
train: 27,675 rows | test: 2,325 rows | test base rate (declining): 0.391


## 3. Train + compare vs my baseline

Same split, same metrics, one table — plus the base rate. The baseline is my ML-07 transparent rule
(0.40 visibility + 0.30 staleness + 0.20 position opportunity + 0.10 thin gap).

In [ ]:
import json, pandas as pd
res = json.load(open('/tmp/ml07/capstone/model_results.json'))['results']
rows = []
for name, r in res.items():
    rows.append([name, r.get('accuracy','--'), r.get('precision','--'), r.get('recall','--'),
                 r.get('f1','--'), r.get('roc_auc','--'), r['p@20'], r['p@50'], r['p@100'], r['base_rate']])
t = pd.DataFrame(rows, columns=['model','acc','prec','rec','f1','auc','p@20','p@50','p@100','base'])
print(t.to_string(index=False))

              model     acc    prec     rec      f1     auc  p@20  p@50  p@100  base
      baseline_rule       —       —       —       —       —  0.10  0.28   0.35 0.391
logistic_regression  0.6606  0.5659  0.5666  0.5662  0.7003  0.35  0.40   0.44 0.391
      decision_tree  0.6766  0.5686  0.7162  0.6339  0.7415  0.40  0.50   0.59 0.391
      random_forest  0.6692  0.5556  0.7701  0.6455  0.7469  0.85  0.76   0.73 0.391


## 4. Errors and interpretation

**What the errors look like** (random forest, test split — 769 of 2,325 wrong at the 0.5 threshold):

- Thin pages with almost no impressions get flagged declining — too little signal to judge either way.
- Stable mid pages score 0.6+ — the model sees risk where the trend label says calm.
- Climbing pages (trend=up) score 0.6 — momentum the proxy label ignores.

**What it leans on** (top features, all plausible — no leakage red flags):
`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`, `word_count`.
No id, no product flag, no future window in the top list. Suspiciously perfect = probably leakage;
nothing here is suspiciously perfect.

**Also read this week:** the FlyRank research paper (March 2026). Two findings I'm auditing in ML-09:
the 3.2x refresh health boost (Finding #4) and the age-freshness matrix (Finding #8) — both get
respectful methodology questions next week.

## 5. Self-check

- [x] Method chosen to fit the ranking question; complexity earned by the comparison
- [x] Baseline in the same table, same split, same metrics, plus base rate
- [x] Errors read before scores believed; top features named and sanity-checked
- [x] Seeds fixed (42); notebook runs top to bottom
- [x] No client names, URLs, or private queries; careful words only